In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
import requests

url = "https://raw.githubusercontent.com/abhishek-marathe04/makemore/refs/heads/main/Indian_names_kaggle.txt"
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    words = response.text.replace(".", "").lower().splitlines()
    print(f"Downloaded {len(words)} names.")
    # print(words[:100])  # Show first 100 characters
else:
    print("Failed to download the file.")

Downloaded 55689 names.


In [5]:
# Build the vocabulary of characters and mappings to/from integers
# print(sorted(list(set(''.join(words)))))
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)

print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [7]:
# Building a Dataset

import torch
import random

block_size = 3

def build_dataset(words):
  print(len(words))
  X = []
  Y = []
  for w in words:

    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)

      # print(''.join(itos[i] for i in context) ,' -->', itos[ix])
      context = context[1:] + [ix]


  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y


random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

44551
5569
5569


In [9]:
# MLP Revisted

n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647) # for reproducability
C = torch.randn(vocab_size, n_embd,             generator=g, requires_grad=True)
W1 = torch.randn(n_embd * block_size, n_hidden ,generator=g, requires_grad=True)
b1 = torch.randn(n_hidden,                      generator=g, requires_grad=True)
W2 = torch.randn(n_hidden, vocab_size,          generator=g, requires_grad=True)
b2 = torch.randn(vocab_size,                    generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]
# print("Model initilised..")
print(f"Hidden Layers : {n_hidden}, embedding_dims : {n_embd}, input_values {vocab_size}, output_values: {vocab_size}")
print(f"Number of parameters in model {sum(p.nelement() for p in parameters)}") # Number of parameters in model

for p in parameters:
    p.requires_grad = True

Hidden Layers : 200, embedding_dims : 10, input_values 27, output_values: 27
Number of parameters in model 11897
